In [58]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta 

In [59]:
df_bruto = pd.read_csv('./vendas.csv', sep=',') 
#print(f"Dataset gerado com {len(df_bruto)} registros.") 
#print(df_bruto.head())

In [60]:
def inspecionar_dados(df_bruto): 
    """Exibe informações básicas do DataFrame.""" 
    print("\n=== INSPEÇÃO INICIAL DO DATASET ===") 
    print(f"Shape: {df_bruto.shape}") 
    print(f"\nColunas: {list(df_bruto.columns)}") 
    print(f"\nTipos de dados:\n{df_bruto.dtypes}") 
    print(f"\nValores nulos por coluna:\n{df_bruto.isnull().sum()}") 
    print(f"\nPrimeiros registros:\n{df_bruto.head()}") 
    print(f"\nEstatísticas descritivas:\n{df_bruto.describe()}") 

In [61]:
#dados_brutos = inspecionar_dados(df_bruto)

In [62]:
import re 
 
def limpar_dados(df_bruto): 
    """ 
    Limpa e trata o DataFrame de vendas. 
    Retorna o DataFrame limpo e um relatório de limpeza. 
    """ 
    n_inicial = len(df_bruto) 
    relatorio = {} 
 
    # 1. Remover espaços extras em colunas de texto 
    colunas_texto = df_bruto.select_dtypes(include=["object", "string"]).columns
    for col in colunas_texto: 
        df_bruto[col] = df_bruto[col].str.strip()

    # 2. Usar regex para limpar caracteres especiais em 'cliente'
    if "cliente" in df_bruto.columns:
        df_bruto["cliente"] = df_bruto["cliente"].apply(lambda x: re.sub(r"[^A-Za-z0-9_]", "", str(x)))
 
    # 3. Converter data e remover datas inválidas 
    df_bruto["data_venda"] = pd.to_datetime(df_bruto["data_venda"], errors="coerce") 
    n_datas_invalidas = df_bruto["data_venda"].isnull().sum() 
    df_bruto = df_bruto.dropna(subset=["data_venda"]) 
    relatorio["datas_invalidas_removidas"] = n_datas_invalidas 
 
    # 4. Remover linhas com quantidade ou preço nulos 
    n_antes = len(df_bruto) 
    df_bruto = df_bruto.dropna(subset=["quantidade", "preco_unitario"]) 
    relatorio["linhas_nulas_removidas"] = n_antes - len(df_bruto) 
 
    # 5. Garantir tipos numéricos corretos 
    df_bruto["quantidade"] = df_bruto["quantidade"].astype(int) 
    df_bruto["preco_unitario"] = df_bruto["preco_unitario"].astype(float) 
 
    n_final = len(df_bruto) 
    relatorio["registros_iniciais"] = n_inicial 
    relatorio["registros_finais"] = n_final 
    relatorio["registros_removidos_total"] = n_inicial - n_final 
 
    print("\n=== RELATÓRIO DE LIMPEZA ===") 
    for chave, valor in relatorio.items(): 
        print(f"  {chave}: {valor}")

    return df_bruto, relatorio

In [63]:
#df_limpo, relatorio = limpar_dados(df_bruto)


In [64]:
def criar_colunas_derivadas(df_limpo): 
    """Cria colunas calculadas e derivadas a partir do dataset limpo.""" 
 
    # Receita total por linha de venda 
    df_limpo["receita_total"] = df_limpo["quantidade"] * df_limpo["preco_unitario"] 
 
    # Extração de componentes de data 
    df_limpo["mes"] = df_limpo["data_venda"].dt.month 
    df_limpo["mes_nome"] = df_limpo["data_venda"].dt.strftime("%B")  # nome do mês 
    df_limpo["trimestre"] = df_limpo["data_venda"].dt.quarter.apply(lambda q: f"Q{q}") 
    df_limpo["ano"] = df_limpo["data_venda"].dt.year 
 
    # Classificação da receita por item com numpy.select (transformação condicional vetorizada) 
    condicoes = [ 
        df_limpo["receita_total"] < 500, 
        (df_limpo["receita_total"] >= 500) & (df_limpo["receita_total"] < 5000), 
        df_limpo["receita_total"] >= 5000 
        ] 
    classificacoes = ["Baixo Valor", "Médio Valor", "Alto Valor"] 
    df_limpo["faixa_receita_item"] = np.select(condicoes, classificacoes, default="Não Classificado")

    print("\n=== COLUNAS DERIVADAS CRIADAS ===") 
    print(df_limpo[["data_venda", "receita_total", "mes", "trimestre", "faixa_receita_item"]].head()) 
    return df_limpo

In [65]:
#dados_apply = criar_colunas_derivadas(df_limpo)


In [66]:
def calcular_metricas(df_limpo): 
    """Calcula e retorna métricas agregadas do dataset.""" 
    metricas = {} 
    # Receita por mês 
    por_mes = df_limpo.groupby("mes").agg( receita_total=("receita_total", "sum"), 
        quantidade=("quantidade", "sum"), 
        n_vendas=("id_venda", "count") 
    ).reset_index().sort_values("mes") 
    metricas["receita por mes"] = por_mes 
 
    # Top 5 produtos por receita 
    top_produtos = df_limpo.groupby("produto")["receita_total"].sum().sort_values(ascending=False).head(5).reset_index() 
    metricas["top 5 produtos por receita"] = top_produtos 
 
    # Receita por categoria 
    por_categoria = df_limpo.groupby("categoria")["receita_total"].sum().reset_index() 
    metricas["receita por categoria"] = por_categoria 
 
    # Receita por região 
    por_regiao = df_limpo.groupby("regiao").agg( 
        receita_total=("receita_total", "sum"), 
        media_ticket=("receita_total", "mean") 
    ).reset_index().sort_values("receita_total", ascending=False) 
    metricas["receita por regiao"] = por_regiao 
 
    # Exibição 
    for nome, tabela in metricas.items(): 
        print(f"\n=== {nome.upper().replace('_', ' ')} ===") 
        print(tabela.to_string(index=False)) 
 
    return metricas

In [67]:
#resultado_metricas = calcular_metricas(df_limpo)

In [68]:
def segmentar_clientes(df_limpo): 
    """Segmenta clientes pelo total gasto usando groupby e lambda.""" 
    clientes = df_limpo.groupby("cliente")["receita_total"].sum().reset_index() 
    clientes.columns = ["cliente", "total_gasto"] 

    # Classificação usando função lambda com condicionais 
    clientes["segmento"] = clientes["total_gasto"].apply( 
        lambda gasto: "Ouro" if gasto > 15000 
        else ("Prata" if gasto >= 5000 else "Bronze") 
        ) 
    clientes = clientes.sort_values("total_gasto", ascending=False)

    print("\n=== SEGMENTAÇÃO DE CLIENTES ===") 
    print(clientes.head(10).to_string(index=False)) 
    print(f"\nDistribuição de segmentos:\n{clientes['segmento'].value_counts()}") 

    return clientes

In [69]:
#clientes_segmentados = segmentar_clientes(df_limpo)

In [70]:
def calcular_estatisticas_numpy(df_limpo): 
    """Usa NumPy para calcular estatísticas sobre as receitas.""" 
    print("\n=== ESTATÍSTICAS COM NUMPY ===") 
 
    receitas = df_limpo["receita_total"].to_numpy()  # Converte para array NumPy 
 
    media = np.mean(receitas) 
    mediana = np.median(receitas) 
    desvio_padrao = np.std(receitas) 
    total = np.sum(receitas) 
    p25 = np.percentile(receitas, 25) 
    p75 = np.percentile(receitas, 75) 
 
    print(f"  Receita média por venda:    R$ {media:.2f}") 
    print(f"  Receita mediana por venda:  R$ {mediana:.2f}") 
    print(f"  Desvio padrão:              R$ {desvio_padrao:.2f}") 
    print(f"  Receita total:              R$ {total:.2f}") 
    print(f"  Percentil 25 (Q1):          R$ {p25:.2f}") 
    print(f"  Percentil 75 (Q3):          R$ {p75:.2f}") 
 
    # Broadcasting: normalizar receitas entre 0 e 1 
    receitas_normalizadas = (receitas - receitas.min()) / (receitas.max() - receitas.min()) 
    print(f"\n  Receitas normalizadas (primeiros 5): {receitas_normalizadas[:5].round(4)}") 
 
    # Operação vetorizada: identificar vendas acima da média sem loop 
    acima_da_media = receitas[receitas > media] 
    print(f"\n  Vendas acima da média: {len(acima_da_media)} de {len(receitas)}") 
 
    return { 
        "media": media, "mediana": mediana, 
        "desvio_padrao": desvio_padrao, "total": total 
    }

In [71]:
#estatisticas_np = calcular_estatisticas_numpy(df_limpo)

In [72]:
import matplotlib.pyplot as plt 
import seaborn as sns 
import os 

def gerar_visualizacoes(df_limpo, resultado_metricas, output_dir="C:/Users/lugul/Downloads/SCTEC/Mini-Projeto/outputs/graficos"): 
    """Gera e exporta visualizações dos dados de vendas.""" 
    os.makedirs(output_dir, exist_ok=True)

    # Configurações visuais globais 
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams["figure.figsize"] = (12, 6)
    plt.rcParams["axes.titlesize"] = 14
    plt.rcParams["axes.labelsize"] = 12

    # --- Gráfico 1: Receita por Mês (linha) --- 
    fig, ax = plt.subplots() 
    por_mes = resultado_metricas["receita por mes"] 
    ax.plot(por_mes["mes"], por_mes["receita_total"], marker="o", 
    linewidth=2, color="#2196F3") 
    ax.fill_between(por_mes["mes"], por_mes["receita_total"], alpha=0.15, 
    color="#2196F3") 
    ax.set_title("Receita Total por Mês (2024)") 
    ax.set_xlabel("Mês") 
    ax.set_ylabel("Receita Total (R$)") 
    ax.set_xticks(range(1, 13)) 
    ax.set_xticklabels(["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"],rotation=45) 
    plt.tight_layout() 
    caminho = os.path.join(output_dir, "vendas_por_mes.png") 
    plt.savefig(caminho, dpi=150) 
    plt.close() 
    print(f"  Gráfico exportado: {caminho}") 
 
    # --- Gráfico 2: Top 5 Produtos (barras horizontais) --- 
    fig, ax = plt.subplots() 
    top = resultado_metricas["top 5 produtos por receita"] 
    sns.barplot(data=top, y="produto", x="receita_total", hue="produto", ax=ax, palette="Blues_d", legend=False)
    ax.set_title("Top 5 Produtos por Receita Total") 
    ax.set_xlabel("Receita Total (R$)") 
    ax.set_ylabel("Produto") 
    for container in ax.containers: 
        ax.bar_label(container, fmt="R$ %.0f", padding=5) 
    plt.tight_layout() 
    caminho = os.path.join(output_dir, "top_5_produtos_por_receita.png") 
    plt.savefig(caminho, dpi=150) 
    plt.close() 
    print(f"  Gráfico exportado: {caminho}") 
 
    # --- Gráfico 3: Distribuição de Receita por Região (boxplot) --- 
    fig, ax = plt.subplots() 
    sns.boxplot(data=df_limpo, x="regiao", y="receita_total", hue="regiao", ax=ax, palette="Set2") 
    ax.set_title("Distribuição de Receita por Transação – Por Região") 
    ax.set_xlabel("Região") 
    ax.set_ylabel("Receita por Venda (R$)") 
    plt.xticks(rotation=30) 
    plt.tight_layout() 
    caminho = os.path.join(output_dir, "distribuicao_regioes.png") 
    plt.savefig(caminho, dpi=150) 
    plt.close() 
    print(f"  Gráfico exportado: {caminho}") 
 
    print("\n=== VISUALIZAÇÕES GERADAS COM SUCESSO ===")

In [73]:
#gerar_visualizacoes(df_limpo, resultado_metricas)


In [74]:
import json

class AnalisadorDeVendas: 
    """ 
    Classe responsável por encapsular o pipeline de análise de vendas. 
    Mantém o estado do DataFrame e os resultados intermediários. 
    """ 
 
    def __init__(self, caminho_arquivo): 
        """Inicializa o analisador com o caminho do arquivo de dados.""" 
        self.caminho_arquivo = caminho_arquivo 
        self.df_bruto = None 
        self.df_limpo = None 
        self.metricas = {} 
        self.clientes = None 
        self.relatorio_limpeza = {} 
 
    def carregar(self): 
        """Lê o arquivo CSV e armazena o DataFrame bruto.""" 
        self.df_bruto = pd.read_csv(self.caminho_arquivo) 
        print(f"[AnalisadorDeVendas] Arquivo carregado: {self.caminho_arquivo}") 
        print(f"  Registros carregados: {len(self.df_bruto)}")

        return self 
 
    def limpar(self): 
        """Limpa os dados e armazena o DataFrame tratado.""" 
        self.df_limpo, self.relatorio_limpeza = limpar_dados(self.df_bruto.copy())

        return self 
 
    def transformar(self): 
        """Aplica transformações e cria colunas derivadas.""" 
        self.df_limpo = criar_colunas_derivadas(self.df_limpo)

        return self 
 
    def analisar(self): 
        """Calcula métricas e segmentações.""" 
        self.metricas = calcular_metricas(self.df_limpo) 
        self.clientes = segmentar_clientes(self.df_limpo) 
        calcular_estatisticas_numpy(self.df_limpo)

        return self 
 
    def visualizar(self): 
        """Gera e exporta os gráficos.""" 
        gerar_visualizacoes(self.df_limpo, self.metricas)

        return self 
 
    def exportar_relatorio(self, caminho="outputs/relatorio_resumo.csv"): 
        """Exporta o relatório de métricas por mês em CSV.""" 
        os.makedirs("outputs", exist_ok=True)

        # Exportar CSV
        self.metricas["receita por mes"].to_csv(caminho, index=False)
        print(f"\n[AnalisadorDeVendas] Relatório exportado: {caminho}")

         # Exportar JSON
        caminho_json = "outputs/relatorio_resumo.json"
        with open(caminho_json, "w", encoding="utf-8") as f:
            json.dump(self.metricas["receita por mes"].to_dict(orient="records"), f, indent=4, ensure_ascii=False)
        print(f"[AnalisadorDeVendas] Relatório exportado: {caminho_json}")

        return self 
 
    def resumo(self): 
        """Exibe um resumo executivo do pipeline.""" 
        print("\n" + "="*50) 
        print("       RESUMO EXECUTIVO – SALESINSIGHT PY") 
        print("="*50) 
        print(f"  Arquivo analisado:      {self.caminho_arquivo}") 
        print(f"  Registros brutos:       {self.relatorio_limpeza.get('registros_iniciais', 'N/A')}") 
        print(f"  Registros limpos:       {self.relatorio_limpeza.get('registros_finais', 'N/A')}") 
        receita = self.df_limpo["receita_total"].sum() if self.df_limpo is not None else 0 
        print(f"  Receita total anual:    R$ {receita:,.2f}") 
        if self.clientes is not None: 
            top = self.clientes.iloc[0] 
            print(f"  Cliente top:            {top['cliente']} (R$ {top['total_gasto']:,.2f})") 
        print("="*50)

In [75]:
class AnalisadorComProjecao(AnalisadorDeVendas): 
    """ 
    Extensão do AnalisadorDeVendas com funcionalidades de projeção simples. 
    Herda todos os métodos da classe pai e adiciona projeção de tendência. 
    """ 
 
    def __init__(self, caminho_arquivo, meses_projecao=3): 
        super().__init__(caminho_arquivo) 
        self.meses_projecao = meses_projecao 
        self.projecoes = [] 
 
    def projetar_tendencia(self): 
        """ 
        Projeta a receita dos próximos meses com base na média móvel dos últimos 3 meses. 
        Método simples sem machine learning – baseado em médias. 
        """ 
        if not self.metricas or "receita por mes" not in self.metricas: 
            print("[AVISO] Rode .analisar() antes de projetar.") 
            return self 
 
        por_mes = self.metricas["receita por mes"].sort_values("mes") 
        receitas_historicas = por_mes["receita_total"].to_numpy() 
 
        # Média móvel dos últimos 3 meses como base da projeção 
        ultimos_3 = receitas_historicas[-3:] 
        media_movel = np.mean(ultimos_3) 
        tendencia = np.std(ultimos_3) * 0.1  # fator de crescimento simples 
 
        ultimo_mes = int(por_mes["mes"].max()) 
 
        print("\n=== PROJEÇÃO DE TENDÊNCIA (Média Móvel Simples) ===")

        print(f"  Base: média dos últimos 3 meses = R$ {media_movel:,.2f}") 
        self.projecoes = [] 
 
        for i in range(1, self.meses_projecao + 1): 
            mes_projetado = (ultimo_mes + i - 1) % 12 + 1 
            receita_projetada = media_movel + (tendencia * i) 
            self.projecoes.append({"mes": mes_projetado, "receita_projetada": round(receita_projetada, 2)}) 
            print(f"  Mês {mes_projetado:02d} (projeção): R$ {receita_projetada:,.2f}") 
 
        return self 
 
    def exibir_projecao_detalhada(self): 
        """Exibe as projeções calculadas.""" 
        if not self.projecoes: 
            print("[AVISO] Nenhuma projeção disponível. Rode .projetar_tendencia() primeiro.") 
            return 
        print("\n=== DETALHAMENTO DAS PROJEÇÕES ===") 
        for p in self.projecoes: 
            print(f"  Mês {p['mes']:02d}: R$ {p['receita_projetada']:,.2f}")

In [76]:
def main(): 
    """ 
    Função principal: executa o pipeline completo do SalesInsight PY. 
    """ 
    print("\n" + "="*60) 
    print("   SALESINSIGHT PY – Pipeline de Análise de Dados de Vendas") 
    print("="*60) 
 
    # Etapa 0: Gerar dataset (se necessário) 
    """if not os.path.exists("vendas.csv"): 
        print("\n[INFO] Gerando dataset sintético...")
    
    df_gerado = gerar_dataset_vendas(n_registros=200) 
    df_gerado.to_csv("vendas.csv", index=False)"""
 
    # Etapa 1 a 6: Pipeline via classe com herança 
    analisador = AnalisadorComProjecao("vendas.csv", meses_projecao=3) 
    (analisador 
        .carregar() 
        .limpar() 
        .transformar() 
        .analisar() 
        .projetar_tendencia() 
        .visualizar() 
        .exportar_relatorio() 
    ) 
 
    """"# Etapa extra: limpeza com regex
    analisador.df_limpo = limpar_strings_com_regex(analisador.df_limpo)
 
    # Etapa extra: exportação JSON
    stats = calcular_estatisticas_numpy(analisador.df_limpo)
    exportar_resultados(analisador.metricas, analisador.clientes, stats)"""
 
    # Resumo final 
    analisador.resumo() 
    analisador.exibir_projecao_detalhada() 
 
    print("\n[CONCLUÍDO] Pipeline finalizado com sucesso!") 
 
 
if __name__ == "__main__": 
    main()


   SALESINSIGHT PY – Pipeline de Análise de Dados de Vendas
[AnalisadorDeVendas] Arquivo carregado: vendas.csv
  Registros carregados: 200

=== RELATÓRIO DE LIMPEZA ===
  datas_invalidas_removidas: 4
  linhas_nulas_removidas: 16
  registros_iniciais: 200
  registros_finais: 180
  registros_removidos_total: 20

=== COLUNAS DERIVADAS CRIADAS ===
  data_venda  receita_total  mes trimestre faixa_receita_item
0 2024-05-20         205.80    5        Q2        Baixo Valor
1 2024-02-17        1504.16    2        Q1        Médio Valor
3 2024-06-21       10007.04    6        Q2         Alto Valor
4 2024-07-12         970.40    7        Q3        Médio Valor
5 2024-04-26        3800.48    4        Q2        Médio Valor

=== RECEITA POR MES ===
 mes  receita_total  quantidade  n_vendas
   1       47169.34          45         9
   2       69968.85          91        15
   3      124532.20          88        17
   4      103611.52          75        15
   5      106602.03          89        14
   6